# LeetCode 1881 — Maximum Value after Insertion bằng Python

Link đề: https://leetcode.com/problems/maximum-value-after-insertion/ · Độ khó: **Medium** · Chủ đề: String, Greedy

Skeleton LeetCode:

```python
class Solution:
    def maxValue(self, n: str, x: int) -> str:
        ...
```

Lộ trình:
1. **Thử mọi vị trí chèn** và so sánh bằng `int(...)` — đúng nhưng chỉ dùng cho chuỗi ngắn (số lớn tới `10⁵` chữ số!).
2. **Thử mọi vị trí** nhưng so sánh **chuỗi cùng độ dài** (không đổi sang số) — `O(L²)`.
3. **Tham lam một lượt**: số dương ⇒ chèn trước chữ số đầu tiên **nhỏ hơn** `x`; số âm ⇒ chèn trước chữ số đầu tiên **lớn hơn** `x` — `O(L)`.
4. Hai cách viết gọn của tham lam: dùng `next(...)` với biểu thức sinh, và xây chuỗi bằng `list` + `join` một lượt.

## 1. Mục tiêu học

1. So sánh hai số nguyên có **cùng số chữ số** bằng **so sánh từ điển** trên chuỗi — kỹ thuật tránh đổi số rất lớn sang `int`.
2. Phân tích ảnh hưởng của **dấu âm**: với số âm, "lớn nhất" ⇔ giá trị tuyệt đối **nhỏ nhất** ⇒ quy tắc tham lam đảo chiều.
3. Chứng minh tham lam "vị trí chèn đầu tiên làm chữ số tăng/giảm" bằng lập luận **vị trí khác nhau đầu tiên**.
4. Viết lời giải `O(L)` một lượt, xử lý biên: chèn ở **cuối** khi không có vị trí nào phù hợp; không chèn trước dấu `'-'`.
5. Kiểm chứng tham lam bằng vét cạn trên mọi vị trí chèn với chuỗi ngẫu nhiên.

## 2. Đề bài & ví dụ

Cho một số nguyên rất lớn `n` dưới dạng **chuỗi** và một chữ số `x`. Các chữ số của `n` và `x` đều thuộc `[1, 9]`; `n` có thể **âm** (bắt đầu bằng `'-'`).

Bạn muốn làm giá trị của `n` **lớn nhất** bằng cách **chèn** `x` vào **một vị trí bất kỳ** trong biểu diễn thập phân của `n`. Không được chèn `x` vào **bên trái** dấu `'-'`.

- `n = 73`, `x = 6` ⇒ tốt nhất chèn giữa 7 và 3 ⇒ `763`.
- `n = −55`, `x = 2` ⇒ tốt nhất chèn trước chữ 5 đầu tiên ⇒ `−255`.

Trả về chuỗi biểu diễn giá trị lớn nhất sau khi chèn.

**Ví dụ 1:** `n = "99"`, `x = 9` → `"999"` (chèn đâu cũng như nhau).

**Ví dụ 2:** `n = "-13"`, `x = 2` → `"-123"` (các khả năng `{-213, -123, -132}`, lớn nhất là `-123`).

**Ràng buộc:**
- `1 ≤ n.length ≤ 10⁵`, `1 ≤ x ≤ 9`
- Các chữ số của `n` thuộc `[1, 9]` (không có chữ số 0)
- `n` là số hợp lệ; nếu âm thì bắt đầu bằng `'-'`.

In [1]:
import random
import time
import sys
from typing import List

examples = [
    (("99", 9), "999"),
    (("-13", 2), "-123"),
    (("73", 6), "763"),
    (("-55", 2), "-255"),
]
for (n, x), want in examples:
    print(f"n={n!r}, x={x} -> kỳ vọng {want!r}")

n='99', x=9 -> kỳ vọng '999'
n='-13', x=2 -> kỳ vọng '-123'
n='73', x=6 -> kỳ vọng '763'
n='-55', x=2 -> kỳ vọng '-255'


## 3. Kiến thức nền

### 3.1 So sánh chuỗi chữ số cùng độ dài

Với hai chuỗi chữ số **cùng độ dài**, so sánh từ điển `a < b` trùng với so sánh giá trị số vì `'0' < '1' < … < '9'`. (Khác độ dài thì không đúng: `"9" > "10"` theo từ điển.)

In [2]:
print("763" > "736", int("763") > int("736"))
print("9" > "10", "(sai lệch khi khác độ dài)")

True True
True (sai lệch khi khác độ dài)


### 3.2 Chèn ký tự vào chuỗi bằng slicing

Chuỗi Python bất biến; chèn tại vị trí `i`: `s[:i] + c + s[i:]` — tốn `O(L)` vì tạo chuỗi mới.

In [3]:
s = "73"
print([s[:i] + "6" + s[i:] for i in range(len(s) + 1)])

['673', '763', '736']


### 3.3 Giới hạn chuyển chuỗi → `int` của Python 3.11+

Python 3.11 trở lên giới hạn mặc định **4300 chữ số** khi đổi chuỗi sang `int` (chống tấn công từ chối dịch vụ). Với `n` dài `10⁵` chữ số, `int(n)` sẽ báo `ValueError`. Ta bắt ngoại lệ để minh họa.

In [4]:
big = "7" * 5000
try:
    int(big)
    print("đổi được (giới hạn đã tắt)")
except ValueError as e:
    print("ValueError:", str(e)[:70], "...")
print("giới hạn hiện tại:", sys.get_int_max_str_digits() if hasattr(sys, "get_int_max_str_digits") else "không có")

ValueError: Exceeds the limit (4300 digits) for integer string conversion: value h ...
giới hạn hiện tại: 4300


### 3.4 `next` với biểu thức sinh và giá trị mặc định

`next((i for i, ch in enumerate(s) if cond(ch)), default)` trả về chỉ số đầu tiên thỏa điều kiện, hoặc `default` nếu không có — gọn cho "tìm vị trí đầu tiên".

In [5]:
s = "9735"
print(next((i for i, ch in enumerate(s) if ch < "6"), len(s)), next((i for i, ch in enumerate(s) if ch < "1"), len(s)))

2 4


## 4. Ý tưởng nền tảng / trực giác

**Số dương** `n = "4639"`, `x = 5`. Liệt kê các vị trí chèn:

| vị trí | kết quả |
|---|---|
| 0 | **54639** |
| 1 | 45639 |
| 2 | 46539 |
| 3 | 46359 |
| 4 | 46395 |

Lớn nhất là `54639` — chèn ngay trước `4`, chữ số đầu tiên nhỏ hơn `5`. Quy luật: các kết quả **cùng độ dài**, nên số lớn nhất là chuỗi lớn nhất theo từ điển. Đi từ trái sang phải: chừng nào chữ số hiện tại `≥ x`, giữ nguyên nó ở vị trí đó là tốt hơn (hoặc bằng) đặt `x` vào; tại chữ số đầu tiên **nhỏ hơn** `x`, đặt `x` vào đó làm vị trí này **tăng** ⇒ tối ưu.

**Số âm** `n = "-4639"`, `x = 5`: muốn giá trị tuyệt đối **nhỏ nhất** ⇒ chèn trước chữ số đầu tiên **lớn hơn** `x`.

In [6]:
def all_insertions(n, x):
    start = 1 if n[0] == "-" else 0
    return [n[:i] + str(x) + n[i:] for i in range(start, len(n) + 1)]

for n, x in [("4639", 5), ("-4639", 5)]:
    cands = all_insertions(n, x)
    print(n, x, "->", cands, "| lớn nhất:", max(cands, key=int))

4639 5 -> ['54639', '45639', '46539', '46359', '46395'] | lớn nhất: 54639
-4639 5 -> ['-54639', '-45639', '-46539', '-46359', '-46395'] | lớn nhất: -45639


## 5. Cách 1 — Brute Force / baseline: thử mọi vị trí, so sánh bằng `int`

**Ý tưởng:** liệt kê mọi chuỗi sau khi chèn (không chèn trước `'-'`), đổi sang `int`, lấy lớn nhất. Đơn giản nhất — nhưng chỉ chạy được với chuỗi ngắn (giới hạn `int` + chi phí chuyển đổi).

In [7]:
def max_value_int(n, x):
    return max(all_insertions(n, x), key=int)

for (n, x), want in examples:
    print(max_value_int(n, x), "kỳ vọng", want)

999 kỳ vọng 999
-123 kỳ vọng -123
763 kỳ vọng 763
-255 kỳ vọng -255


**Độ phức tạp:** `L + 1` ứng viên × `O(L)` tạo chuỗi × chuyển `int` (`O(L²)` với thuật toán chuyển đổi thông thường) ⇒ rất chậm với `L = 10⁵`, và vướng giới hạn 4300 chữ số.

### 5.1 Cách 1b — Thử mọi vị trí, so sánh **chuỗi** (không đổi sang số)

Mọi ứng viên cùng độ dài `L + 1` và cùng dấu ⇒
- số dương: lớn nhất ⇔ chuỗi lớn nhất theo từ điển;
- số âm: lớn nhất ⇔ phần chữ số (sau `'-'`) **nhỏ nhất** theo từ điển.

In [8]:
def max_value_strcmp(n, x):
    cands = all_insertions(n, x)
    if n[0] == "-":
        return min(cands)          # cùng tiền tố '-', cùng độ dài: chuỗi nhỏ nhất = trị tuyệt đối nhỏ nhất
    return max(cands)

for (n, x), want in examples:
    print(max_value_strcmp(n, x), "kỳ vọng", want)

999 kỳ vọng 999
-123 kỳ vọng -123
763 kỳ vọng 763
-255 kỳ vọng -255


**Độ phức tạp:** `O(L²)` thời gian và bộ nhớ tạm (mỗi ứng viên `O(L)`). Với `L = 10⁵` là `10¹⁰` — vẫn quá chậm.

**Nút thắt:** hầu hết ứng viên khác nhau ở **một** vị trí "chèn"; ta không cần tạo chúng — chỉ cần tìm vị trí tốt nhất bằng một lần quét.

In [9]:
for L in [1000, 4000]:
    n = "".join(random.choice("123456789") for _ in range(L))
    t0 = time.perf_counter(); max_value_strcmp(n, 5)
    print(f"L={L}: {time.perf_counter() - t0:.3f}s")

L=1000: 0.001s
L=4000: 0.008s


## 6. Ý tưởng thuật toán / bước đột phá

Xét số dương với chữ số `d₀ d₁ … d_{L−1}`. Chèn `x` ở vị trí `i` cho chuỗi `S_i = d₀ … d_{i−1} x d_i … d_{L−1}`.

**So sánh `S_i` và `S_{i+1}`:** hai chuỗi trùng nhau ở `i` ký tự đầu; ở vị trí `i`, `S_i` có `x` còn `S_{i+1}` có `d_i`. Nếu `x ≠ d_i` thì vị trí này quyết định:
- `d_i > x` ⇒ `S_{i+1} > S_i` (chèn muộn hơn tốt hơn).
- `d_i < x` ⇒ `S_i > S_{i+1}`.
- `d_i = x` ⇒ `S_i = S_{i+1}` (hoán đổi hai chữ số giống nhau).

**Insight:** gọi `p` là chỉ số đầu tiên có `d_p < x` (hoặc `p = L` nếu không có). Với `i < p`: `d_i ≥ x` ⇒ `S_{i+1} ≥ S_i`. Với `i ≥ p`: … ta chứng minh `S_p ≥ S_j` cho mọi `j > p` ở mục 10 (vì `S_p` đã "thắng" ở vị trí `p`). ⇒ **chèn tại `p`**.

**Số âm:** giá trị `−|v|` lớn nhất ⇔ `|v|` nhỏ nhất ⇒ lặp lại lập luận với chiều ngược: chèn trước chữ số **đầu tiên lớn hơn** `x` (bỏ qua dấu `'-'`).

**Quy tắc nếu… thì…:**
- `n` dương: duyệt `i` từ 0; nếu `n[i] < x` thì chèn tại `i`; hết chuỗi thì chèn ở cuối.
- `n` âm: duyệt `i` từ 1; nếu `n[i] > x` thì chèn tại `i`; hết chuỗi thì chèn ở cuối.
- Bằng nhau (`n[i] == x`) ⇒ **phải đi tiếp** (chèn tại đây chỉ bằng chèn ngay sau nó, nhưng phía sau có thể còn chữ số lớn hơn để vượt qua).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tham lam một lượt

In [10]:
def max_value_greedy(n, x):
    c = str(x)
    if n[0] == "-":
        for i in range(1, len(n)):
            if n[i] > c:
                return n[:i] + c + n[i:]
    else:
        for i in range(len(n)):
            if n[i] < c:
                return n[:i] + c + n[i:]
    return n + c

for (n, x), want in examples:
    print(max_value_greedy(n, x), "kỳ vọng", want)

999 kỳ vọng 999
-123 kỳ vọng -123
763 kỳ vọng 763
-255 kỳ vọng -255


So sánh `n[i] > c` là so sánh **ký tự** — đúng vì cả hai là chữ số đơn.

**Độ phức tạp:** quét `O(L)` + tạo chuỗi kết quả `O(L)` ⇒ Time `O(L)`, Space `O(L)` cho kết quả.

### 7.2 Cách 3 — Viết gọn bằng `next(...)`

In [11]:
def max_value_next(n, x):
    c = str(x)
    if n[0] == "-":
        i = next((i for i in range(1, len(n)) if n[i] > c), len(n))
    else:
        i = next((i for i in range(len(n)) if n[i] < c), len(n))
    return n[:i] + c + n[i:]

for (n, x), want in examples:
    print(max_value_next(n, x), "kỳ vọng", want)

999 kỳ vọng 999
-123 kỳ vọng -123
763 kỳ vọng 763
-255 kỳ vọng -255


**Độ phức tạp:** như Cách 2.

### 7.3 Cách 4 — Xây chuỗi một lượt bằng danh sách ký tự

Không dùng slicing: duyệt từng ký tự, khi gặp vị trí cần chèn thì thêm `x` **một lần** (cờ `inserted`), cuối cùng nếu chưa chèn thì thêm vào cuối. Cách này thuận tiện khi dữ liệu đến dạng **luồng** (đọc từng ký tự).

In [12]:
def max_value_stream(n, x):
    c = str(x)
    neg = n[0] == "-"
    out = []
    inserted = False
    for i, ch in enumerate(n):
        if not inserted and not (neg and i == 0):
            if (neg and ch > c) or (not neg and ch < c):
                out.append(c)
                inserted = True
        out.append(ch)
    if not inserted:
        out.append(c)
    return "".join(out)

for (n, x), want in examples:
    print(max_value_stream(n, x), "kỳ vọng", want)

999 kỳ vọng 999
-123 kỳ vọng -123
763 kỳ vọng 763
-255 kỳ vọng -255


**Độ phức tạp:** Time `O(L)`, Space `O(L)`.

### 7.4 Cách 5 — Quy số âm về số dương bằng "đảo chữ số"

Mẹo thống nhất hai trường hợp: với số âm, thay mỗi chữ số `d` bằng `10 − d` (và `x` bằng `10 − x`), áp quy tắc **số dương**, rồi đổi ngược lại. Vì phép `d ↦ 10 − d` đảo thứ tự các chữ số `1..9`, "chữ số đầu tiên lớn hơn `x`" trở thành "chữ số đầu tiên nhỏ hơn `10 − x`".

In [13]:
def max_value_mirror(n, x):
    if n[0] != "-":
        return max_value_greedy(n, x)
    flip = str.maketrans("123456789", "987654321")     # d -> 10 - d
    body = n[1:].translate(flip)
    res = max_value_greedy(body, 10 - x)                # áp quy tắc số dương
    return "-" + res.translate(flip)

for (n, x), want in examples:
    print(max_value_mirror(n, x), "kỳ vọng", want)

999 kỳ vọng 999
-123 kỳ vọng -123
763 kỳ vọng 763
-255 kỳ vọng -255


`str.maketrans` + `translate` đổi mọi chữ số trong `O(L)`. Cách này cho thấy hai quy tắc thực chất là **một**.

## 8. TRACE từng bước

Trace Cách 2 với số dương `n = "4639"`, `x = 5` và số âm `n = "-4639"`, `x = 5`:

| i | n[i] | dương: `n[i] < 5`? | âm: `n[i] > 5`? |
|---|---|---|---|
| 0 / 1 | 4 | **có** ⇒ chèn ở 0 ⇒ `54639` | không |
| 1 / 2 | 6 | — | **có** ⇒ chèn trước 6 ⇒ `-45639` |

In [14]:
def trace_greedy(n, x):
    c = str(x); neg = n[0] == "-"
    start = 1 if neg else 0
    print(f"n={n}, x={x}, {'âm: tìm chữ số đầu tiên > x' if neg else 'dương: tìm chữ số đầu tiên < x'}")
    for i in range(start, len(n)):
        hit = n[i] > c if neg else n[i] < c
        print(f"  i={i}: n[i]={n[i]} -> {'CHÈN tại đây' if hit else 'đi tiếp'}")
        if hit:
            res = n[:i] + c + n[i:]; print("  kết quả:", res); return res
    res = n + c; print("  không gặp -> chèn ở cuối:", res); return res

trace_greedy("4639", 5)
trace_greedy("-4639", 5)
trace_greedy("999", 9)
trace_greedy("-111", 1)

n=4639, x=5, dương: tìm chữ số đầu tiên < x
  i=0: n[i]=4 -> CHÈN tại đây
  kết quả: 54639
n=-4639, x=5, âm: tìm chữ số đầu tiên > x
  i=1: n[i]=4 -> đi tiếp
  i=2: n[i]=6 -> CHÈN tại đây
  kết quả: -45639
n=999, x=9, dương: tìm chữ số đầu tiên < x
  i=0: n[i]=9 -> đi tiếp
  i=1: n[i]=9 -> đi tiếp
  i=2: n[i]=9 -> đi tiếp
  không gặp -> chèn ở cuối: 9999
n=-111, x=1, âm: tìm chữ số đầu tiên > x
  i=1: n[i]=1 -> đi tiếp
  i=2: n[i]=1 -> đi tiếp
  i=3: n[i]=1 -> đi tiếp
  không gặp -> chèn ở cuối: -1111


'-1111'

In [15]:
def trace_compare_neighbors(n, x):
    """Minh họa so sánh S_i và S_{i+1} ở mục 6 cho số dương."""
    c = str(x)
    S = [n[:i] + c + n[i:] for i in range(len(n) + 1)]
    for i in range(len(n)):
        rel = ">" if S[i] > S[i + 1] else ("<" if S[i] < S[i + 1] else "=")
        print(f"S_{i}={S[i]} {rel} S_{i + 1}={S[i + 1]}   (d_{i}={n[i]} so với x={c})")

trace_compare_neighbors("7536", 5)

S_0=57536 < S_1=75536   (d_0=7 so với x=5)
S_1=75536 = S_2=75536   (d_1=5 so với x=5)
S_2=75536 > S_3=75356   (d_2=3 so với x=5)
S_3=75356 < S_4=75365   (d_3=6 so với x=5)


Quan sát: chuỗi `S_0, S_1, …` **tăng (không giảm)** cho tới vị trí `p` đầu tiên có `d_p < x`, rồi tại `p` "rơi" xuống. Cực đại nằm ở `S_p`.

## 9. Code nộp LeetCode

In [16]:
class Solution:
    def maxValue(self, n: str, x: int) -> str:
        c = str(x)
        if n[0] == "-":
            for i in range(1, len(n)):
                if n[i] > c:
                    return n[:i] + c + n[i:]
        else:
            for i in range(len(n)):
                if n[i] < c:
                    return n[:i] + c + n[i:]
        return n + c

sol = Solution()
for (n, x), want in examples:
    got = sol.maxValue(n, x)
    print(got, "OK" if got == want else "SAI")

999 OK
-123 OK
763 OK
-255 OK


## 10. Vì sao thuật toán đúng

**Số dương.** Gọi `p` = chỉ số đầu tiên với `d_p < x` (hoặc `L`). Ta chứng minh `S_p ≥ S_j` với mọi `j`.

- **`j < p`:** với mọi `i < p` có `d_i ≥ x` ⇒ `S_{i+1} ≥ S_i` (mục 6). Bắc cầu: `S_j ≤ S_{j+1} ≤ … ≤ S_p`.
- **`j > p`:** `S_p` và `S_j` trùng nhau ở `p` ký tự đầu (`d₀ … d_{p−1}`). Ở vị trí `p`: `S_p` có `x`, `S_j` có `d_p < x` ⇒ `S_p > S_j`.

Mọi ứng viên cùng độ dài ⇒ so sánh từ điển = so sánh giá trị ⇒ `S_p` lớn nhất.

**Số âm.** Mọi ứng viên có dạng `−T` với `T` cùng độ dài ⇒ giá trị lớn nhất ⇔ `T` nhỏ nhất. Lập luận đối xứng: với `q` = chỉ số đầu tiên (trong phần chữ số) có `d_q > x`, các `T_j` không tăng cho tới `q`, và `T_q < T_j` với `j > q` ⇒ `T_q` nhỏ nhất.

**Không có chữ số 0** nên không có trường hợp số 0 đứng đầu làm thay đổi độ dài; và với số âm, không được chèn trước `'-'` nên vòng lặp bắt đầu từ chỉ số 1.

## 11. Phân tích Time / Space

Gọi `L = len(n) ≤ 10⁵`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mọi vị trí + `int` | `O(L³)` (đổi `int` `O(L²)`) | `O(L²)` | chuỗi ngắn, vướng giới hạn 4300 |
| 1b. Mọi vị trí + so chuỗi | `O(L²)` | `O(L²)` | kiểm chứng |
| 2. Tham lam | `O(L)` | `O(L)` | **nộp bài** |
| 3. `next(...)` | `O(L)` | `O(L)` | gọn |
| 4. Xây luồng | `O(L)` | `O(L)` | dữ liệu dạng luồng |
| 5. Đảo chữ số | `O(L)` | `O(L)` | thống nhất hai trường hợp |

In [17]:
random.seed(1)
n_pos = "".join(random.choice("6789") for _ in range(10**5))       # không có chữ số < 5 -> quét hết
n_neg = "-" + "".join(random.choice("1234") for _ in range(10**5))
for name, fn in [("greedy", max_value_greedy), ("next", max_value_next), ("stream", max_value_stream),
                 ("mirror", max_value_mirror), ("submit", Solution().maxValue)]:
    t0 = time.perf_counter(); a = fn(n_pos, 5); b = fn(n_neg, 5)
    print(f"{name:6s}: {time.perf_counter() - t0:.4f}s | chèn ở cuối? {a.endswith('5')}, {b.endswith('5')}")

greedy: 0.0043s | chèn ở cuối? True, True
next  : 0.0043s | chèn ở cuối? True, True
stream: 0.0100s | chèn ở cuối? True, True
mirror: 0.0043s | chèn ở cuối? True, True
submit: 0.0041s | chèn ở cuối? True, True


## 12. Bộ test

Dữ liệu ngẫu nhiên đúng ràng buộc: chữ số `1..9`, có thể có dấu `'-'`. So mọi cách với vét cạn bằng `int` (chuỗi ngắn).

In [18]:
fns = [max_value_strcmp, max_value_greedy, max_value_next, max_value_stream, max_value_mirror,
       lambda n, x: Solution().maxValue(n, x)]

for (n, x), want in examples:
    assert max_value_int(n, x) == want
    for fn in fns:
        assert fn(n, x) == want

edge = [(("1", 9), "91"), (("9", 1), "91"), (("-1", 9), "-19"), (("-9", 1), "-19"),
        (("5", 5), "55"), (("-5", 5), "-55"), (("123", 4), "4123"), (("-987", 1), "-1987"),
        (("-123", 4), "-1234"), (("987", 1), "9871")]
for (n, x), want in edge:
    assert max_value_int(n, x) == want, (n, x)
    for fn in fns:
        assert fn(n, x) == want, (fn, n, x)

random.seed(2024)
for _ in range(3000):
    L = random.randint(1, 12)
    digits = "".join(random.choice(random.choice(["123456789", "59", "1", "9"])) for _ in range(L))
    n = ("-" if random.random() < 0.5 else "") + digits
    x = random.randint(1, 9)
    want = max_value_int(n, x)
    for fn in fns:
        assert fn(n, x) == want, (fn, n, x)
print("ví dụ + edge + 3000 bộ ngẫu nhiên khớp vét cạn bằng int")

ví dụ + edge + 3000 bộ ngẫu nhiên khớp vét cạn bằng int


In [19]:
# chuỗi dài: các cách tham lam khớp nhau và khớp vét cạn so chuỗi
random.seed(8)
for _ in range(50):
    L = random.randint(100, 600)
    n = ("-" if random.random() < 0.5 else "") + "".join(random.choice("123456789") for _ in range(L))
    x = random.randint(1, 9)
    r = max_value_strcmp(n, x)
    assert r == max_value_greedy(n, x) == max_value_next(n, x) == max_value_stream(n, x) == max_value_mirror(n, x)
    assert len(r) == len(n) + 1
print("chuỗi dài tới 600: tham lam khớp vét cạn so chuỗi")

chuỗi dài tới 600: tham lam khớp vét cạn so chuỗi


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng cùng một quy tắc cho số âm và số dương

In [20]:
def wrong_same_rule(n, x):
    c = str(x); start = 1 if n[0] == "-" else 0
    for i in range(start, len(n)):
        if n[i] < c:                                # SAI với số âm
            return n[:i] + c + n[i:]
    return n + c

for n, x in [("-13", 2), ("-55", 2), ("-4639", 5)]:
    print(f"{n}, {x}: sai = {wrong_same_rule(n, x)} | đúng = {max_value_greedy(n, x)}")

-13, 2: sai = -213 | đúng = -123
-55, 2: sai = -552 | đúng = -255
-4639, 5: sai = -54639 | đúng = -45639


**Sửa:** số âm chèn trước chữ số đầu tiên **lớn hơn** `x`.

### Lỗi 2 — Chèn trước dấu `'-'` (vòng lặp bắt đầu từ 0 với số âm)

In [21]:
def wrong_before_sign(n, x):
    c = str(x)
    if n[0] == "-":
        for i in range(0, len(n)):                  # SAI: i = 0 là dấu '-'
            if n[i] > c:
                return n[:i] + c + n[i:]
        return n + c
    return max_value_greedy(n, x)

print(wrong_before_sign("-13", 2), "| đúng =", max_value_greedy("-13", 2))
print("'-' > '2' ?", "-" > "2", "-> dấu '-' (mã 45) nhỏ hơn mọi chữ số nên lỗi tình cờ không lộ")

def wrong_before_sign_min(n, x):          # cùng lỗi, nhưng ở bài toán "giá trị nhỏ nhất"
    c = str(x)
    for i in range(0, len(n)):            # SAI: i = 0 là dấu '-'
        if n[i] < c:
            return n[:i] + c + n[i:]
    return n + c
print("biến thể 'nhỏ nhất' với cùng lỗi:", wrong_before_sign_min("-13", 2), "-> chuỗi không còn là số hợp lệ!")

-123 | đúng = -123
'-' > '2' ? False -> dấu '-' (mã 45) nhỏ hơn mọi chữ số nên lỗi tình cờ không lộ
biến thể 'nhỏ nhất' với cùng lỗi: 2-13 -> chuỗi không còn là số hợp lệ!


Trong bản trên, `'-' > '2'` là `False` nên lỗi "may mắn" không lộ — một lỗi **tiềm ẩn**. Với điều kiện ngược (`<`, như khi tìm giá trị nhỏ nhất), `'-' < '2'` là `True` ⇒ `x` bị chèn **trước** dấu, tạo chuỗi `"2-13"` không hợp lệ. **Sửa:** bắt đầu từ chỉ số 1 với số âm.

### Lỗi 3 — Quên chèn ở cuối khi không tìm thấy vị trí

In [22]:
def wrong_no_append(n, x):
    c = str(x)
    for i in range(len(n)):
        if n[i] < c:
            return n[:i] + c + n[i:]
    return n                                        # SAI: chưa chèn x

for n, x in [("99", 9), ("987", 1)]:
    print(f"{n}, {x}: sai = {wrong_no_append(n, x)!r} | đúng = {max_value_greedy(n, x)!r}")

99, 9: sai = '99' | đúng = '999'
987, 1: sai = '987' | đúng = '9871'


**Sửa:** `return n + c`.

### Lỗi 4 — Dùng `<=` thay vì `<` (dừng ở chữ số **bằng** `x`)

Nghe có vẻ vô hại: chèn `x` trước hay sau một chữ số bằng `x` cho cùng chuỗi (`S_i = S_{i+1}` khi `d_i = x`). Nhưng dừng **sớm** tại chữ số bằng nhau khiến ta bỏ lỡ một chữ số **lớn hơn** phía sau — nơi lẽ ra nên đi tiếp.

In [23]:
def wrong_le(n, x):
    c = str(x)
    if n[0] == "-":
        i = next((i for i in range(1, len(n)) if n[i] >= c), len(n))   # SAI: phải là >
    else:
        i = next((i for i in range(len(n)) if n[i] <= c), len(n))      # SAI: phải là <
    return n[:i] + c + n[i:]

for n, x in [("57", 5), ("-53", 5), ("4639", 5)]:
    print(f"{n}, {x}: sai = {wrong_le(n, x)} | đúng = {max_value_greedy(n, x)}")

57, 5: sai = 557 | đúng = 575
-53, 5: sai = -553 | đúng = -535
4639, 5: sai = 54639 | đúng = 54639


`"57"`, `x = 5`: bản lỗi dừng ở `'5'` ⇒ `"557"`, nhưng đi tiếp qua `'5'` rồi `'7'` (cả hai `≥ 5`) và chèn ở cuối cho `"575"` lớn hơn. Ví dụ `"4639"` không lộ lỗi — vì vậy phải thử các trường hợp có chữ số **bằng** `x`. **Sửa:** dương dùng `<` (nghiêm ngặt), âm dùng `>` (nghiêm ngặt).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Số rất lớn dạng chuỗi ⇒ **không** đổi sang `int`; so sánh chuỗi **cùng độ dài** theo từ điển.
- "Chèn/xóa/đổi một chữ số để lớn nhất" ⇒ tìm **vị trí đầu tiên** mà thao tác làm chữ số tại đó **tăng** (tham lam theo vị trí khác nhau đầu tiên).
- Số âm ⇒ đảo mục tiêu (nhỏ nhất theo trị tuyệt đối), hoặc đảo chữ số `d ↦ 10 − d`.
- Họ bài: 670 *Maximum Swap*, 402 *Remove K Digits*, 1323 *Maximum 69 Number*, 2259 *Remove Digit From Number to Maximize Result*.

## 15. Biến thể của bài

1. **Giá trị nhỏ nhất sau khi chèn:** đảo vai trò hai quy tắc (dương: chèn trước chữ số đầu tiên **lớn hơn** `x`).
2. **Chèn `k` chữ số `x`:** chèn cả `k` chữ số liền nhau tại cùng vị trí tham lam (vì mỗi lần chèn, vị trí tối ưu không đổi).
3. **Chèn một chữ số tùy chọn** (được chọn `x`): luôn chọn `x = 9` cho số dương (chèn ở đầu).
4. **Có chữ số 0 và cấm số 0 đứng đầu:** thêm điều kiện không chèn `0` ở vị trí đầu.
5. **Xóa một chữ số để lớn nhất** (bài khác): xóa chữ số đầu tiên mà chữ số sau nó lớn hơn.

In [24]:
def min_value_after_insertion(n, x):
    """Biến thể 1."""
    c = str(x)
    if n[0] == "-":
        i = next((i for i in range(1, len(n)) if n[i] < c), len(n))
    else:
        i = next((i for i in range(len(n)) if n[i] > c), len(n))
    return n[:i] + c + n[i:]

def insert_k_times(n, x, k):
    """Biến thể 2."""
    for _ in range(k):
        n = max_value_greedy(n, x)
    return n

def insert_k_brute(n, x, k):
    cands = {n}
    for _ in range(k):
        cands = {s for t in cands for s in all_insertions(t, x)}
    return max(cands, key=int)

random.seed(5)
for _ in range(500):
    n = ("-" if random.random() < .5 else "") + "".join(random.choice("123456789") for _ in range(random.randint(1, 6)))
    x = random.randint(1, 9)
    assert min_value_after_insertion(n, x) == min(all_insertions(n, x), key=int)
    k = random.randint(1, 3)
    assert insert_k_times(n, x, k) == insert_k_brute(n, x, k)
print("biến thể 1, 2 khớp vét cạn")

biến thể 1, 2 khớp vét cạn


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi vị trí + `int` | định nghĩa | rất chậm | `O(L²)` | kiểm chứng chuỗi ngắn |
| 1b. Mọi vị trí + so chuỗi | cùng độ dài ⇒ từ điển | `O(L²)` | `O(L²)` | kiểm chứng |
| 2. Tham lam | vị trí khác nhau đầu tiên | `O(L)` | `O(L)` | **nộp bài** |
| 3. `next(...)` | như 2 | `O(L)` | `O(L)` | viết gọn |
| 4. Xây luồng | thêm `x` một lần | `O(L)` | `O(L)` | đọc dạng luồng |
| 5. Đảo chữ số | âm ⇒ dương | `O(L)` | `O(L)` | hiểu bản chất |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `maxValue("469975787", 8)` và `maxValue("-648468153646", 5)`?

**Bài 2 (tính tay):** liệt kê mọi kết quả chèn `x = 3` vào `n = "-2525"` và chọn lớn nhất.

**Bài 3 (điền TODO):** hoàn thiện `best_position(n, x)` trả về **chỉ số** chèn (trong chuỗi gốc) theo quy tắc tham lam.

**Bài 4 (tự viết lại):** viết `max_value_one_rule(n, x)` dùng **một** vòng lặp duy nhất cho cả hai dấu, nhờ hàm so sánh `better(ch)` chọn theo dấu.

**Bài 5 (mini challenge):** được chèn **một chữ số bất kỳ trong tập `D`** (ví dụ `D = {2, 7}`) đúng một lần. Viết hàm trả về giá trị lớn nhất và kiểm chứng bằng vét cạn. Gợi ý: với số dương, chữ số lớn nhất trong `D` luôn tốt nhất? Còn số âm? (Biến thể tự đặt.)

In [25]:
def best_position(n, x):
    # TODO: trả về i sao cho n[:i] + str(x) + n[i:] là kết quả tối ưu
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) Số dương, tìm chữ số đầu tiên `< 8`. (2) `4 < 8` ngay đầu ⇒ chèn ở 0. (3) `"8469975787"`; số âm: chữ số đầu tiên `> 5` là `6` ở đầu ⇒ `"-5648468153646"`.

**Bài 2:** (1) Các vị trí 1..5. (2) `-32525, -23525, -25325, -25235, -25253`. (3) Lớn nhất `-23525` (chèn trước chữ số đầu tiên `> 3`, tức `5`).

**Bài 3:** (1) Như Cách 3 nhưng trả về `i`. (2) Âm bắt đầu từ 1. (3) Không gặp ⇒ `len(n)`.

**Bài 4:** (1) `neg = n[0] == '-'`. (2) `better = (lambda ch: ch > c) if neg else (lambda ch: ch < c)`. (3) Vòng lặp từ `start = int(neg)`.

**Bài 5:** (1) Số dương: thử chèn `max(D)` theo tham lam — so sánh với chèn chữ số khác bằng vét cạn. (2) Số âm: chữ số **nhỏ nhất** trong `D` chưa chắc… hãy thử. (3) An toàn: thử mọi `d ∈ D` với tham lam, lấy kết quả tốt nhất (`|D|` lượt `O(L)`).

## 19. Đáp án tham khảo

In [26]:
# Bài 1
assert Solution().maxValue("469975787", 8) == "8469975787"
assert Solution().maxValue("-648468153646", 5) == "-5648468153646"

# Bài 2
cands = all_insertions("-2525", 3)
print(cands, "->", max(cands, key=int))
assert max(cands, key=int) == "-23525" == Solution().maxValue("-2525", 3)

# Bài 3
def best_position_ans(n, x):
    c = str(x)
    if n[0] == "-":
        return next((i for i in range(1, len(n)) if n[i] > c), len(n))
    return next((i for i in range(len(n)) if n[i] < c), len(n))

# Bài 4
def max_value_one_rule(n, x):
    c = str(x); neg = n[0] == "-"
    better = (lambda ch: ch > c) if neg else (lambda ch: ch < c)
    for i in range(int(neg), len(n)):
        if better(n[i]):
            return n[:i] + c + n[i:]
    return n + c

random.seed(141)
for _ in range(2000):
    n = ("-" if random.random() < .5 else "") + "".join(random.choice("123456789") for _ in range(random.randint(1, 8)))
    x = random.randint(1, 9)
    i = best_position_ans(n, x)
    assert n[:i] + str(x) + n[i:] == max_value_int(n, x) == max_value_one_rule(n, x)
print("Bài 1-4: OK")

['-32525', '-23525', '-25325', '-25235', '-25253'] -> -23525
Bài 1-4: OK


In [27]:
# Bài 5 — chèn một chữ số bất kỳ trong tập D
def max_value_from_set(n, D):
    return max((max_value_greedy(n, d) for d in D), key=int)

def max_value_from_set_brute(n, D):
    return max((s for d in D for s in all_insertions(n, d)), key=int)

random.seed(142)
pos_ok = neg_ok = True
for _ in range(2000):
    n = ("-" if random.random() < .5 else "") + "".join(random.choice("123456789") for _ in range(random.randint(1, 7)))
    D = random.sample(range(1, 10), random.randint(1, 4))
    assert max_value_from_set(n, D) == max_value_from_set_brute(n, D)
    if n[0] != "-" and max_value_greedy(n, max(D)) != max_value_from_set_brute(n, D):
        pos_ok = False
    if n[0] == "-" and max_value_greedy(n, min(D)) != max_value_from_set_brute(n, D):
        neg_ok = False
print("thử mọi d trong D: khớp vét cạn")
print("số dương: chỉ cần max(D)?", pos_ok, "| số âm: chỉ cần min(D)?", neg_ok)

thử mọi d trong D: khớp vét cạn
số dương: chỉ cần max(D)? True | số âm: chỉ cần min(D)? True


Nhận xét Bài 5 (theo output): thử mọi `d ∈ D` luôn đúng. Với số dương, chèn `max(D)` luôn tối ưu (một chữ số lớn hơn đặt ở vị trí tham lam của nó cho chuỗi không nhỏ hơn); với số âm, `min(D)` cũng tối ưu theo lập luận đối xứng — output in `True` cho cả hai xác nhận điều này trên 2000 bộ.

## 20. Template nhớ nhanh + Checklist

```python
c = str(x)
if n[0] == "-":                                   # âm: trị tuyệt đối nhỏ nhất
    for i in range(1, len(n)):
        if n[i] > c: return n[:i] + c + n[i:]
else:                                             # dương: lớn nhất
    for i in range(len(n)):
        if n[i] < c: return n[:i] + c + n[i:]
return n + c                                      # không gặp: chèn ở cuối
```

**Checklist trước khi submit:**
- [ ] Dương: chèn trước chữ số đầu tiên **< x**; âm: trước chữ số đầu tiên **> x**.
- [ ] Số âm: vòng lặp bắt đầu từ chỉ số 1 (không chèn trước `'-'`).
- [ ] Không tìm thấy ⇒ chèn ở cuối.
- [ ] Không đổi chuỗi sang `int` (dài tới `10⁵` chữ số).
- [ ] So sánh ký tự với `str(x)`, không phải với số `x`.